# Module 09: Machine Learning with scikit-learn

Follow along here during class. Feel free to add your own cells and notes.

The git section at the start of today's class (`git pull`, `git rebase`) happens in your terminal, not in this notebook.

In [ ]:
# Only needed if you run this notebook in the browser (JupyterLite).
%pip install -q formulaic

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from formulaic import model_matrix

df = pd.read_csv('data/gapminder.tsv', sep='\t')
df = df.rename(columns={'lifeExp': 'life_expectancy', 'gdpPercap': 'gdp_per_capita'})
df['log_gdp'] = np.log10(df['gdp_per_capita'])
df['log_pop'] = np.log10(df['pop'])
df_2007 = df.query("year == 2007")

## scikit-learn

### Features and Target

In [ ]:
X = df_2007[['log_gdp']]
y = df_2007['life_expectancy']

### Fit and Predict

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)

predictions = model.predict(X)

In [ ]:
model.intercept_, model.coef_

### The Same Line as statsmodels

Fit the same model with `smf.ols` from Module 8, as we did before:

In [ ]:
import statsmodels.formula.api as smf

ols_model = smf.ols('life_expectancy ~ log_gdp', data=df_2007)
results = ols_model.fit()
print(results.summary())

`results.params` is the equivalent of reading the coefficients from the summary. Compare the intercept and slope with scikit-learn. Fit on all the data, they are identical.

Recap: `smf.ols` with a formula adds an intercept automatically. With scikit-learn we use the columns directly, and it fits an intercept automatically too.

In [ ]:
results.params

### Train and Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Fit on the Train Set

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

In [ ]:
model.intercept_, model.coef_

### Evaluating the Model

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

print(r2_score(y_test, predictions))
print(mean_squared_error(y_test, predictions) ** 0.5)

### Practice (5 min)

Let's work on these questions:

1. Train a `LinearRegression` with both `log_gdp` and `log_pop` as features. Use `X = df_2007[['log_gdp', 'log_pop']]`.
2. Does the test R² go up compared to using `log_gdp` alone?
3. Change `random_state` in `train_test_split` to another number. How much does the test R² change? What does that tell you about a small test set?

In [ ]:
# 1.

In [ ]:
# 2.

In [ ]:
# 3.

### Classification

In Module 8 we asked, with a logistic regression in statsmodels: does a country have a high life expectancy, meaning above the median in 2007? Let's ask it again with scikit-learn.

In [ ]:
median = df_2007['life_expectancy'].median()
df_2007 = df_2007.assign(high_life_exp=(df_2007['life_expectancy'] > median).astype(int))
df_2007['high_life_exp'].value_counts()

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

X = df_2007[['log_gdp']]
y = df_2007['high_life_exp']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression()
clf.fit(X_train, y_train)
clf.predict(X_test)

### Accuracy and the Confusion Matrix

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

predictions = clf.predict(X_test)
print(accuracy_score(y_test, predictions))
confusion_matrix(y_test, predictions)

### Practice (5 min)

Let's work on these questions:

1. Add `continent` as a feature. Use `model_matrix('~ log_gdp + C(continent)', df_2007)` which returns just `X` since the formula has no outcome, and drop its `Intercept` column.
2. Does the accuracy go up?
3. Use `clf.predict_proba(X_test)` to see the probabilities behind the predictions. Which test country was the model least sure about?

In [ ]:
# 1.

In [ ]:
# 2.

In [ ]:
# 3.